# Update Existing Residential Units - Parcel_History_Attributed

**Goal:** Compare CSV source data against `SDE.Parcel_History_Attributed`, validate differences, identify missing APNs, then update `Residential_Units`.

**Source CSV:** `data/raw_data/ExistingResidential_2012_2025_unstacked.csv`  
**Target FC:** `SDE.Parcels\SDE.Parcel_History_Attributed` (Vector.sde)

## 1. Setup

In [ ]:
import os
import pathlib
import warnings

import pandas as pd
import numpy as np
import arcpy

warnings.filterwarnings("ignore")

local_path = pathlib.Path().absolute()

# Feature class — work directly on this
PARCEL_GDB  = r"C:/GIS/ParcelHistory.gdb"
fc_path     = PARCEL_GDB + "/Parcel_History_Attributed"
LOCAL_FC    = fc_path   # alias used throughout notebook
SCRATCH_GDB = PARCEL_GDB  # scratch centroid FCs land here
assert arcpy.Exists(fc_path), f"Feature class not found: {fc_path}"
print(f"Feature class : {fc_path}")

# SDE connection — used as fallback for spatial join sources if needed
sdeBase = None
for _candidate in [
    pathlib.Path(r"F:/GIS/DB_CONNECT/Vector.sde"),
    pathlib.Path(r"F:/GIS/PARCELUPDATE/Workspace/Vector.sde"),
]:
    if _candidate.exists():
        sdeBase = _candidate
        break

# CSV
csv_path = local_path / "data" / "raw_data" / "ExistingResidential_2012_2025_unstacked.csv"
print(f"CSV           : {csv_path}")
assert csv_path.exists(), f"CSV not found: {csv_path}"

arcpy.env.workspace       = "memory"
arcpy.env.overwriteOutput = True


## 2. Load and Reshape CSV

Wide format (one row per APN) melted to long format (one row per APN x Year).

In [ ]:
df_wide = pd.read_csv(csv_path)

year_cols = [c for c in df_wide.columns if str(c).strip().endswith('Final')]
df_wide   = df_wide[['APN'] + year_cols].copy()
df_wide['APN'] = df_wide['APN'].astype(str).str.strip()

print(f'CSV rows (APNs): {len(df_wide):,}')
print(f'Year columns   : {year_cols}')
df_wide.head()

In [ ]:
df_csv = df_wide.melt(id_vars='APN', var_name='Year_Label', value_name='Residential_Units_CSV')
df_csv['Year'] = df_csv['Year_Label'].str.extract(r'(\d{4})').astype(int)
df_csv = df_csv.drop(columns='Year_Label')
df_csv = df_csv.dropna(subset=['Residential_Units_CSV'])
df_csv['Residential_Units_CSV'] = df_csv['Residential_Units_CSV'].astype(int)

print(f'Long-format rows: {len(df_csv):,}')
print(f'Unique APNs     : {df_csv["APN"].nunique():,}')
print(f'Years covered   : {sorted(df_csv["Year"].unique())}')
df_csv.head(10)

## 3. Load Feature Class into Pandas

In [ ]:
fields = arcpy.ListFields(fc_path)
field_names = [f.name for f in fields]
print('Feature class fields:')
for f in fields:
    print(f'  {f.name:<40} {f.type}')

In [ ]:
# Adjust if field names differ from what is shown above
FC_APN_FIELD    = 'APN'
FC_YEAR_FIELD   = 'YEAR'
FC_RES_FIELD    = 'Residential_Units'
FC_COUNTY_FIELD = 'COUNTY'

for fld in [FC_APN_FIELD, FC_YEAR_FIELD, FC_RES_FIELD, FC_COUNTY_FIELD]:
    assert fld in field_names, f"Field '{fld}' NOT FOUND. Available: {field_names}"
    print(f'  ok {fld}')


In [ ]:
read_fields = [FC_APN_FIELD, FC_YEAR_FIELD, FC_RES_FIELD, FC_COUNTY_FIELD, "OID@"]
print(f'Reading: {fc_path}')
print(f'Fields : {read_fields}')

rows = []
with arcpy.da.SearchCursor(fc_path, read_fields) as cursor:
    for row in cursor:
        rows.append({
            'APN'              : str(row[0]).strip() if row[0] else None,
            FC_YEAR_FIELD      : int(row[1]) if row[1] is not None else None,
            'Residential_Units': row[2],
            'COUNTY'           : row[3],
            'OID'              : row[4],
        })

print(f'Rows from cursor: {len(rows):,}')
if len(rows) == 0:
    raise ValueError(f'No rows returned. Check path: {fc_path}')

df_fc = pd.DataFrame(rows)
df_fc = df_fc.rename(columns={FC_YEAR_FIELD: "Year"})

csv_years = sorted(df_csv["Year"].unique())
df_fc_filtered = df_fc[df_fc["Year"].isin(csv_years)].copy()

print(f'FC total rows       : {len(df_fc):,}')
print(f'FC rows (2012-2025) : {len(df_fc_filtered):,}')
print(f'FC unique APNs      : {df_fc_filtered["APN"].nunique():,}')
df_fc_filtered.head(10)


## 2b. Fix El Dorado County APN Format

El Dorado County changed their APN format by zero-padding the suffix:
- **Old (CSV):**  e.g. 
- **New (FC, later years):**  e.g. 

We detect affected APNs, check which format the FC actually uses per year, and normalize  so the join works correctly.

In [ ]:
import re

# Build APN -> COUNTY lookup from the full FC
apn_county = (
    df_fc.dropna(subset=["APN", "COUNTY"])
    .groupby("APN")["COUNTY"]
    .first()
    .to_dict()
)
el_dorado_apns = {apn for apn, c in apn_county.items() if c == "EL"}
print(f'Unique APNs in FC           : {len(apn_county):,}')
print(f'El Dorado (COUNTY=EL) APNs  : {len(el_dorado_apns):,}')

# Show county breakdown
display(
    df_fc.dropna(subset=["APN","COUNTY"])
    .drop_duplicates("APN")[["APN","COUNTY"]]
    .groupby("COUNTY").size()
    .reset_index(name="unique_APNs")
    .sort_values("unique_APNs", ascending=False)
)

# Regex: APN with 2-digit suffix -> candidate for zero-padding
apn_suffix_2digit = re.compile(r"^(\d{3}-\d{2,3})-(\d{2})$")

def pad_suffix(apn):
    m = apn_suffix_2digit.match(apn)
    return f"{m.group(1)}-0{m.group(2)}" if m else None

# El Dorado APNs in the CSV that have a 2-digit suffix
candidates = {
    apn for apn in df_csv["APN"].unique()
    if apn_suffix_2digit.match(apn) and apn in el_dorado_apns
}
print(f'El Dorado APNs to pad (Year>=2018): {len(candidates):,}')


In [ ]:
EL_DORADO_PAD_YEAR = 2018  # El Dorado added leading zero to APN suffix from this year

padded_map = {apn: pad_suffix(apn) for apn in candidates}

df_csv["APN_original"] = df_csv["APN"].copy()

df_csv["APN"] = df_csv.apply(
    lambda row: padded_map[row["APN"]]
    if row["APN"] in padded_map and row["Year"] >= EL_DORADO_PAD_YEAR
    else row["APN"],
    axis=1
)

changed = df_csv["APN"] != df_csv["APN_original"]
print(f'Rows updated in df_csv         : {changed.sum():,}')
print(f'Unique El Dorado APNs padded   : {df_csv.loc[changed, "APN_original"].nunique():,}')
print(f'Years affected                 : {sorted(df_csv.loc[changed, "Year"].unique())}')

df_csv[changed][["APN_original","APN","Year","Residential_Units_CSV"]].head(20)


## 4. Validate - Compare CSV vs Feature Class

In [ ]:
# APN-level comparison
csv_apns = set(df_csv['APN'].unique())
fc_apns  = set(df_fc_filtered['APN'].dropna().unique())

apns_only_in_csv = csv_apns - fc_apns   # APN never appears in FC at all
apns_only_in_fc  = fc_apns  - csv_apns  # APN in FC but not in CSV

print(f"APNs in CSV              : {len(csv_apns):,}")
print(f"APNs in FC (2012-2025)   : {len(fc_apns):,}")
print(f"APNs only in CSV         : {len(apns_only_in_csv):,}  <- never appear in FC")
print(f"APNs only in FC          : {len(apns_only_in_fc):,}")

# APN x Year-level comparison
csv_apn_year = set(zip(df_csv['APN'], df_csv['Year']))
fc_apn_year  = set(zip(df_fc_filtered['APN'].fillna(''), df_fc_filtered['Year'].fillna(-1)))

missing_apn_year = csv_apn_year - fc_apn_year   # in CSV but not FC
extra_apn_year   = fc_apn_year  - csv_apn_year  # in FC but not CSV

print(f"APN x Year combinations:")
print(f"  In CSV                 : {len(csv_apn_year):,}")
print(f"  In FC                  : {len(fc_apn_year):,}")
print(f"  Missing from FC        : {len(missing_apn_year):,}  <- rows that need to be added")
print(f"  In FC not in CSV       : {len(extra_apn_year):,}")

In [ ]:
# Build a DataFrame of all missing APN x Year combinations
df_missing_apn_year = pd.DataFrame(sorted(missing_apn_year), columns=["APN", "Year"])

# Tag whether the APN exists in the FC at all (just missing that year)
# vs. the APN is completely absent from the FC
df_missing_apn_year['apn_in_fc_other_years'] = df_missing_apn_year['APN'].isin(fc_apns)

completely_absent = df_missing_apn_year[~df_missing_apn_year["apn_in_fc_other_years"]]
partial_missing   = df_missing_apn_year[ df_missing_apn_year["apn_in_fc_other_years"]]

absent_apns = df_missing_apn_year.loc[
    ~df_missing_apn_year["apn_in_fc_other_years"], "APN"
].nunique()
missing_year_apns = df_missing_apn_year.loc[
    df_missing_apn_year["apn_in_fc_other_years"], "APN"
].nunique()

print(f"Missing APN x Year rows total    : {len(df_missing_apn_year):,}")
print(f"APN completely absent from FC  : {absent_apns:,} APNs ({len(completely_absent):,} rows)")
print(f"APN exists but year is missing : {missing_year_apns:,} APNs ({len(partial_missing):,} rows)")

# Breakdown by year
print(f'Missing rows by year:')
display(df_missing_apn_year.groupby('Year').size().reset_index(name='missing_count'))

df_missing_apn_year.sort_values(["APN", "Year"])

In [ ]:
df_compare = df_csv.merge(
    df_fc_filtered[['APN','Year','Residential_Units','OID']],
    on=['APN','Year'],
    how='inner'
)

df_compare['values_match'] = (
    df_compare['Residential_Units_CSV'] == df_compare['Residential_Units']
)
df_diffs = df_compare[~df_compare['values_match']].copy()

print(f'Matched APN x Year rows : {len(df_compare):,}')
print(f'Value mismatches        : {len(df_diffs):,}')
print(f'Already correct         : {df_compare["values_match"].sum():,}')

if len(df_diffs):
    display(df_diffs[['APN','Year','Residential_Units_CSV','Residential_Units']].rename(
        columns={'Residential_Units':'Residential_Units_FC'}
    ).sort_values(['APN','Year']).head(30))

In [ ]:
summary = pd.DataFrame({
    "Category": [
        "APNs in CSV",
        "APNs in FC (filtered to CSV years)",
        "APNs completely absent from FC",
        "APNs in FC not in CSV",
        "APN x Year combos in CSV",
        "APN x Year missing from FC (need add)",
        "  - APN completely absent",
        "  - APN exists but year missing",
        "APN x Year value mismatches (need update)",
        "APN x Year already correct",
    ],
    "Count": [
        len(csv_apns),
        len(fc_apns),
        len(apns_only_in_csv),
        len(apns_only_in_fc),
        len(csv_apn_year),
        len(missing_apn_year),
        len(completely_absent),
        len(partial_missing),
        len(df_diffs),
        int(df_compare["values_match"].sum()),
    ]
})
summary

## 5. Export Validation Results

In [ ]:
out_dir = local_path / "data" / "processed_data"
out_dir.mkdir(parents=True, exist_ok=True)

# All missing APN x Year combinations
missing_out = out_dir / "ResUnits_missing_apn_year.csv"
df_missing_apn_year.sort_values(["APN","Year"]).to_csv(missing_out, index=False)
print(f"Saved missing APN x Year -> {missing_out}")

# Value mismatches for rows that DO exist in both
if len(df_diffs):
    diffs_out = out_dir / "ResUnits_value_mismatches.csv"
    df_diffs[["OID","APN","Year","Residential_Units_CSV","Residential_Units"]].rename(
        columns={"Residential_Units":"Residential_Units_FC"}
    ).sort_values(["APN","Year"]).to_csv(diffs_out, index=False)
    print(f"Saved mismatches        -> {diffs_out}")
else:
    print("No value mismatches - FC already matches CSV for all overlapping APN x Year rows.")

## 6. Update Residential_Units

- All rows for 2012-2025: `Residential_Units = 0` (clears NULLs and stale values)
- Rows matching CSV: updated to the CSV value
- Missing APN x Year rows: inserted with geometry from MapServer query

In [ ]:
import math, json as _json

def safe_int(val):
    if val is None: return 0
    try:
        if math.isnan(float(val)): return 0
    except (TypeError, ValueError): pass
    return int(val)

# (APN, Year) -> CSV value
csv_lookup = {
    (row["APN"], int(row["Year"])): safe_int(row["Residential_Units_CSV"])
    for _, row in df_csv.iterrows()
}

# OID -> new value for every row in csv_years (default 0, override with CSV)
update_dict = {}
for _, row in df_fc_filtered.iterrows():
    key = (row["APN"], int(row["Year"]))
    update_dict[int(row["OID"])] = csv_lookup.get(key, 0)

n_matched = sum(1 for _,r in df_fc_filtered.iterrows()
                if (r["APN"], int(r["Year"])) in csv_lookup)
print(f"FC rows to update         : {len(update_dict):,}")
print(f"  matched to CSV value    : {n_matched:,}")
print(f"  defaulting to 0         : {len(update_dict) - n_matched:,}")

try:
    geom_lookup = {(r["APN"], int(r["Year"])): r["geometry"]
                   for _, r in df_geom.iterrows()}
    print(f"Insert rows (missing APN/Year): {len(df_missing_apn_year):,}")
    print(f"Geometry available            : {len(geom_lookup):,}")
except NameError:
    geom_lookup = {}
    print("Warning: df_geom not found - run MapServer query cells first.")


In [ ]:
print("Updating Residential_Units on local FC...")
updated = 0
with arcpy.da.UpdateCursor(LOCAL_FC, ["OID@", FC_RES_FIELD]) as cursor:
    for row in cursor:
        if row[0] in update_dict:
            row[1] = update_dict[row[0]]
            cursor.updateRow(row)
            updated += 1
print(f"  Updated: {updated:,} rows")
print("Done. Run the MapServer query cells (c23-c26) to fetch geometry")
print("for missing APN x Year rows, review in ArcGIS Pro, then run c27 to insert.")


## 7. Spatial Attribute Updates

Populate spatial attributes for new/null rows.
Pattern mirrors `CountyParcel_Transform.py` in the ParcelUpdate repo:
1. PARCEL_ACRES, PARCEL_SQFT from geometry
2. WITHIN_TRPA_BNDY, WITHIN_BONUSUNIT_BNDY via SelectByLocation
3. Centroid spatial joins: TOWN_CENTER, TAZ, PLAN, ZONING, REGIONAL_LANDUSE

In [ ]:
# ── Spatial join source layers (all confirmed service URLs) ──────────────
SPATIAL_SOURCES = {
    "TRPA_bdy"          : "https://maps.trpa.org/server/rest/services/Boundaries/FeatureServer/4",
    "BonusUnit"         : "https://maps.trpa.org/server/rest/services/Housing/MapServer/8",
    "TownCenter"        : "https://maps.trpa.org/server/rest/services/Boundaries/FeatureServer/1",
    "LocationToTownCtr" : "https://maps.trpa.org/server/rest/services/Planning/FeatureServer/4",
    "TAZ"               : "https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/6",
    "LocalPlan"         : "https://maps.trpa.org/server/rest/services/Planning/FeatureServer/2",
    "Zoning"            : "https://maps.trpa.org/server/rest/services/Zoning/MapServer/0",
    "RegionalLandUse"   : "https://maps.trpa.org/server/rest/services/LocalPlan/MapServer/7",
}

# Scratch feature classes (all local)
ParcelPoint        = SCRATCH_GDB + "/PHA_Points"
ParcelPoint_Town   = SCRATCH_GDB + "/PHA_Town"
ParcelPoint_LocTc  = SCRATCH_GDB + "/PHA_LocTC"
ParcelPoint_TAZ    = SCRATCH_GDB + "/PHA_TAZ"
ParcelPoint_Plan   = SCRATCH_GDB + "/PHA_Plan"
ParcelPoint_Zoning = SCRATCH_GDB + "/PHA_Zoning"
ParcelPoint_RLU    = SCRATCH_GDB + "/PHA_RLU"

def field_join_calc(update_fc, key_fields, value_fields,
                    source_fc, source_key_fields, source_value_fields):
    """Transfer values from source_fc to update_fc via APN key dict.
    Adapted from fieldJoinCalc_multikey() in ParcelUpdate/utils.py.
    """
    value_dict = {}
    for row in arcpy.da.SearchCursor(source_fc, source_key_fields + source_value_fields):
        k = tuple(str(v) if v is not None else "" for v in row[:len(source_key_fields)])
        if any(k):
            value_dict[k] = list(row[len(source_key_fields):])
    updated = 0
    with arcpy.da.UpdateCursor(update_fc, key_fields + value_fields) as cur:
        for row in cur:
            k = tuple(str(v) if v is not None else "" for v in row[:len(key_fields)])
            if k in value_dict:
                cur.updateRow(list(row[:len(key_fields)]) + value_dict[k])
                updated += 1
    return updated

print("Spatial join sources (all service URLs):")
for k, v in SPATIAL_SOURCES.items():
    print(f"  {k:<20} {v}")


In [ ]:
year_list   = ", ".join(str(y) for y in csv_years)
where_years = f"{FC_YEAR_FIELD} IN ({year_list})"
arcpy.management.MakeFeatureLayer(LOCAL_FC, "local_lyr", where_years)
n_scope = int(arcpy.management.GetCount("local_lyr").getOutput(0))
print(f"Rows in scope: {n_scope:,}")

# PARCEL_ACRES and PARCEL_SQFT
print("Calculating PARCEL_ACRES / PARCEL_SQFT...")
n = 0
with arcpy.da.UpdateCursor("local_lyr", ["PARCEL_ACRES", "PARCEL_SQFT", "SHAPE@"]) as cur:
    for row in cur:
        if row[2]:
            row[0] = row[2].getArea("PLANAR", "ACRES")
            row[1] = row[2].getArea("PLANAR", "SquareFeetUS")
            cur.updateRow(row)
            n += 1
print(f"  {n:,} rows")

# Boundary flags
def flag_within(lyr, source_fc, field):
    with arcpy.da.UpdateCursor(lyr, [field]) as cur:
        for row in cur: row[0] = 0; cur.updateRow(row)
    sel = arcpy.management.SelectLayerByLocation(lyr, "INTERSECT", source_fc)
    n = 0
    with arcpy.da.UpdateCursor(sel, [field]) as cur:
        for row in cur: row[0] = 1; cur.updateRow(row); n += 1
    arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")
    return n

n1 = flag_within("local_lyr", SPATIAL_SOURCES["TRPA_bdy"],  "WITHIN_TRPA_BNDY")
n2 = flag_within("local_lyr", SPATIAL_SOURCES["BonusUnit"], "WITHIN_BONUSUNIT_BNDY")
print(f"WITHIN_TRPA_BNDY=1       : {n1:,}")
print(f"WITHIN_BONUSUNIT_BNDY=1  : {n2:,}")


In [ ]:
# Centroid points (created once, used for all joins)
for fc in [ParcelPoint, ParcelPoint_Town, ParcelPoint_LocTc, ParcelPoint_TAZ,
           ParcelPoint_Plan, ParcelPoint_Zoning, ParcelPoint_RLU]:
    if arcpy.Exists(fc): arcpy.management.Delete(fc)

arcpy.management.FeatureToPoint("local_lyr", ParcelPoint, "INSIDE")
print(f"Centroids: {int(arcpy.management.GetCount(ParcelPoint).getOutput(0)):,}")

def sj_transfer(join_src, out_fc, source_fields, target_fields, label):
    arcpy.analysis.SpatialJoin(
        ParcelPoint, join_src, out_fc,
        "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="HAVE_THEIR_CENTER_IN")
    n = field_join_calc(
        "local_lyr", ["APN"], target_fields,
        out_fc,      ["APN"], source_fields)
    print(f"  {label:<35} {n:,} rows")

# TOWN_CENTER (name of the town center polygon)
sj_transfer(SPATIAL_SOURCES["TownCenter"],      ParcelPoint_Town,
            source_fields=["NAME"],
            target_fields=["TOWN_CENTER"],
            label="TOWN_CENTER")

# LOCATION_TO_TOWNCENTER (separate Planning service)
sj_transfer(SPATIAL_SOURCES["LocationToTownCtr"], ParcelPoint_LocTc,
            source_fields=["LOCATION_TO_TOWNCENTER"],  # confirm field name in service
            target_fields=["LOCATION_TO_TOWNCENTER"],
            label="LOCATION_TO_TOWNCENTER")

# TAZ
sj_transfer(SPATIAL_SOURCES["TAZ"],             ParcelPoint_TAZ,
            source_fields=["TAZ"],
            target_fields=["TAZ"],
            label="TAZ")

# PLAN_ID + PLAN_NAME
sj_transfer(SPATIAL_SOURCES["LocalPlan"],       ParcelPoint_Plan,
            source_fields=["PLAN_ID", "PLAN_NAME"],
            target_fields=["PLAN_ID", "PLAN_NAME"],
            label="PLAN_ID / PLAN_NAME")

# ZONING_ID + ZONING_DESCRIPTION
sj_transfer(SPATIAL_SOURCES["Zoning"],          ParcelPoint_Zoning,
            source_fields=["ZONING_ID", "ZONING_DESCRIPTION"],
            target_fields=["ZONING_ID", "ZONING_DESCRIPTION"],
            label="ZONING")

# REGIONAL_LANDUSE
sj_transfer(SPATIAL_SOURCES["RegionalLandUse"], ParcelPoint_RLU,
            source_fields=["REGIONAL_LANDUSE"],
            target_fields=["REGIONAL_LANDUSE"],
            label="REGIONAL_LANDUSE")

for fc in [ParcelPoint, ParcelPoint_Town, ParcelPoint_LocTc, ParcelPoint_TAZ,
           ParcelPoint_Plan, ParcelPoint_Zoning, ParcelPoint_RLU]:
    if arcpy.Exists(fc): arcpy.management.Delete(fc)

print("\nAll spatial attribute updates complete on local copy.")
print(f"Local FC ready to review: {LOCAL_FC}")


## 8. Fetch Missing Geometry from AllParcels MapServer

Service: `https://maps.trpa.org/server/rest/services/AllParcels/MapServer`

One polygon layer per year (2006-2024). For each missing APN x Year we query
the matching layer. 2025 is not yet in the service - those rows fall back to 2024.

Results are written to `C:/GIS/Scratch.gdb` for review **before** anything touches SDE.

In [ ]:
from arcgis.features import FeatureLayer

SERVICE_URL = "https://maps.trpa.org/server/rest/services/AllParcels/MapServer"

# Layer ID per year (from service metadata)
YEAR_LAYER = {
    2024: 32, 2023: 31, 2022: 30, 2021: 29,
    2020: 27, 2019: 22, 2018: 20, 2017: 17, 2016: 18,
    2015:  5, 2014:  6, 2013:  7, 2012:  8,
}
FALLBACK_YEAR = 2024  # use 2024 layer for any year not in dict (e.g. 2025)

def get_layer(year):
    layer_id = YEAR_LAYER.get(year, YEAR_LAYER[FALLBACK_YEAR])
    return FeatureLayer(f"{SERVICE_URL}/{layer_id}")

# Group missing APN x Year by year
missing_by_year = df_missing_apn_year.groupby("Year")["APN"].apply(list).to_dict()

print("Missing APN x Year counts by year:")
for yr, apns in sorted(missing_by_year.items()):
    layer_id = YEAR_LAYER.get(yr, YEAR_LAYER[FALLBACK_YEAR])
    note = "" if yr in YEAR_LAYER else f" (no {yr} layer, using {FALLBACK_YEAR} layer {layer_id})"
    print(f"  {yr}: {len(apns):>5,} APNs  ->  layer {layer_id}{note}")

In [ ]:
BATCH_SIZE = 50

# All available years sorted — used to build nearest-year search order
AVAILABLE_YEARS = sorted(YEAR_LAYER.keys())

def pad_apn(apn):
    import re
    m = re.match(r"^(\d{3}-\d{2,3})-(\d{2})$", apn)
    return f"{m.group(1)}-0{m.group(2)}" if m else None

def nearest_years(target, available=AVAILABLE_YEARS):
    """Return available years sorted by proximity to target year.
    e.g. target=2013 -> [2013, 2014, 2012, 2015, 2011, ...]
    """
    return sorted(available, key=lambda y: (abs(y - target), y))

def query_layer(layer, apns, skip_pad=None):
    """Query a single layer for a list of APNs.
    Returns {csv_apn: feature}. Tries zero-padded suffix if not found (EL county).
    """
    _skip_pad = skip_pad or set()
    found = {}

    def _fetch(apn_list):
        quoted = ",".join(f"'{a}'" for a in apn_list)
        try:
            return layer.query(
                where=f"APN IN ({quoted})",
                out_fields="APN",
                return_geometry=True
            ).features
        except Exception as e:
            print(f"    Warning: {e}")
            return []

    # Pass 1 — original APN format
    for i in range(0, len(apns), BATCH_SIZE):
        for feat in _fetch(apns[i:i + BATCH_SIZE]):
            fc_apn = feat.attributes.get("APN")
            if fc_apn:
                found[fc_apn] = feat

    # Pass 2 — zero-padded retry for El Dorado APNs not yet found
    padded_to_orig = {pad_apn(a): a for a in apns if pad_apn(a)}
    orig_to_padded = {v: k for k, v in padded_to_orig.items()}
    found_orig = (
        set(found.keys())
        | {padded_to_orig[k] for k in found if k in padded_to_orig}
    )
    need_pad = [
        orig_to_padded[a] for a in apns
        if a not in found_orig and a in orig_to_padded and a not in _skip_pad
    ]
    for i in range(0, len(need_pad), BATCH_SIZE):
        for feat in _fetch(need_pad[i:i + BATCH_SIZE]):
            fc_apn = feat.attributes.get("APN")
            if fc_apn and fc_apn in padded_to_orig:
                orig = padded_to_orig[fc_apn]
                found[orig] = feat
                found[orig]._apn_used = fc_apn

    return found


# ── Main fetch loop with nearest-year fallback ───────────────────────────
geom_rows      = []   # {APN, Year, geom_year, geometry}
apn_format_log = []   # El Dorado padding log
still_missing  = []   # APNs not found in ANY year

# Layer cache so we don't re-instantiate for each fallback
_layer_cache = {}
def get_cached_layer(year):
    if year not in _layer_cache:
        _layer_cache[year] = get_layer(year)
    return _layer_cache[year]

print("Querying AllParcels MapServer (with nearest-year fallback)...")

for target_year, apns in sorted(missing_by_year.items()):
    skip_pad = {a for a in apns if apn_county.get(a, "EL") not in ("EL", None, "")}
    remaining = list(apns)  # APNs still needing geometry
    year_order = nearest_years(target_year)

    for search_year in year_order:
        if not remaining:
            break
        layer  = get_cached_layer(search_year)
        result = query_layer(layer, remaining, skip_pad=skip_pad)

        found_this_round = []
        for csv_apn, feat in result.items():
            fc_apn = getattr(feat, "_apn_used", csv_apn)
            geom_rows.append({
                "APN"           : csv_apn,
                "APN_in_service": fc_apn,
                "Year"          : target_year,
                "geom_year"     : search_year,   # which year's layer the geometry came from
                "geometry"      : feat.geometry,
            })
            if fc_apn != csv_apn:
                apn_format_log.append({"Year": target_year, "geom_year": search_year,
                                       "APN_csv": csv_apn, "APN_service": fc_apn})
            found_this_round.append(csv_apn)

        remaining = [a for a in remaining if a not in found_this_round]

        if result and search_year != target_year:
            print(f"  {target_year}: used {search_year} layer for {len(result):,} APNs")

    if remaining:
        still_missing.extend([(a, target_year) for a in remaining])

df_geom = pd.DataFrame(geom_rows)

print(f"\nGeometry fetched   : {len(df_geom):,} rows")
print(f"Still not found    : {len(still_missing):,} rows")

if len(df_geom):
    fallback_used = df_geom[df_geom["geom_year"] != df_geom["Year"]]
    print(f"Fallback year used : {len(fallback_used):,} rows ({len(fallback_used)/len(df_geom)*100:.1f}%)")

if still_missing:
    df_still_missing = pd.DataFrame(still_missing, columns=["APN", "Year"])
    print("\nAPNs not found in any year:")
    display(df_still_missing)

if apn_format_log:
    df_fmt = pd.DataFrame(apn_format_log)
    print(f"\nEl Dorado padding used for {len(df_fmt):,} rows")

df_geom.head()


In [ ]:
# Join geometry to missing APN x Year rows and bring in Residential_Units value
df_to_insert = df_missing_apn_year.merge(
    df_geom,
    on=["APN", "Year"],
    how="left"
).merge(
    df_csv[["APN", "Year", "Residential_Units_CSV"]],
    on=["APN", "Year"],
    how="left"
)

has_geom     = df_to_insert["geometry"].notna()
missing_geom = df_to_insert[~has_geom]

print(f"Rows to insert total       : {len(df_to_insert):,}")
print(f"  With geometry            : {has_geom.sum():,}")
print(f"  Without geometry (check) : {(~has_geom).sum():,}")

if len(missing_geom):
    print("\nRows with no geometry found (may need manual lookup):")
    display(missing_geom[["APN","Year","apn_in_fc_other_years"]].head(20))

In [ ]:
# Write rows WITH geometry to Scratch.gdb for review in ArcGIS Pro
from arcgis.geometry import Geometry

scratch_gdb = pathlib.Path(r"C:/GIS/Scratch.gdb")
out_fc_path = str(scratch_gdb / "ResUnits_to_insert")

df_with_geom = df_to_insert[has_geom].copy().rename(
    columns={"Residential_Units_CSV": "Residential_Units"}
)

# Convert geometry dicts to arcgis Geometry objects
df_with_geom["SHAPE"] = df_with_geom["geometry"].apply(
    lambda g: Geometry(g) if isinstance(g, dict) else g
)
df_with_geom = df_with_geom.drop(columns="geometry")

sdf = df_with_geom.copy()
sdf.spatial.set_geometry("SHAPE")
sdf.spatial.to_featureclass(
    location=out_fc_path,
    overwrite=True,
    sanitize_columns=False
)

print(f"Written {len(df_with_geom):,} rows to:")
print(f"  {out_fc_path}")
print("\nOpen in ArcGIS Pro to review before inserting into SDE.")

## 9. Insert Reviewed Rows into Local GDB

**Only run after reviewing `ResUnits_to_insert` in ArcGIS Pro.**

Appends the new APN x Year rows (with geometry) from `Scratch.gdb` into
`SDE.Parcel_History_Attributed`. Adjust `INSERT_FIELDS` to include any
other non-nullable fields required by the schema.

In [ ]:
# Insert reviewed rows from scratch FC into the local working FC
# Review ResUnits_to_insert in ArcGIS Pro before running this cell.
INSERT_FIELDS = [FC_APN_FIELD, FC_YEAR_FIELD, FC_RES_FIELD, "SHAPE@"]

inserted = 0
with arcpy.da.InsertCursor(LOCAL_FC, INSERT_FIELDS) as icursor:
    with arcpy.da.SearchCursor(out_fc_path, INSERT_FIELDS) as scursor:
        for row in scursor:
            icursor.insertRow(row)
            inserted += 1

print(f"Inserted {inserted:,} rows into {LOCAL_FC}")
